In [1]:
import numpy as np
import scanpy as sc
import pyvips
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import tqdm.auto as tqdm
import os

In [2]:
format_to_dtype = {
    'uchar': np.uint8,
    'char': np.int8,
    'ushort': np.uint16,
    'short': np.int16,
    'uint': np.uint32,
    'int': np.int32,
    'float': np.float32,
    'double': np.float64,
    'complex': np.complex64,
    'dpcomplex': np.complex128,
}

def crop_tile(image, x_pixel, y_pixel, cell_diameter):
    x = x_pixel - int(cell_diameter // 2)
    y = y_pixel - int(cell_diameter // 2)
    cell = image.crop(y, x, cell_diameter, cell_diameter)
    main_tile = np.ndarray(buffer=cell.write_to_memory(),
                           dtype=format_to_dtype[cell.format],
                           shape=[cell.height, cell.width, cell.bands])
    main_tile = main_tile[:, :, :3]
    return main_tile

In [9]:
adata_path = snakemake.input.h5ad
adata = sc.read_h5ad(adata_path)

AnnData object with n_obs × n_vars = 4361 × 4659
    obs: 'in_tissue', 'x_array', 'y_array', 'x_pixel', 'y_pixel', 'manual_anno', 'manual_anno_tls', 'aestetik_manual_anno', 'aestetik_manual_anno_tls', 'ground_truth', 'blur_score', 'sampleID', 'barcode'
    uns: 'enrichr_INFL', 'enrichr_NOR', 'enrichr_TLS', 'enrichr_TUM', 'enrichr_UNASSIGNED', 'he_slide', 'meta', 'spatial'
    obsm: 'pathways_decoupler_progeny_human', 'ref_atlas_anno', 'spatial'
    layers: 'raw_counts'

In [70]:
slide = pyvips.Image.new_from_array(adata.uns['he_slide'])

In [ ]:
os.makedirs(snakemake.output.cropped_dir, exist_ok=True)

In [84]:
if snakemake.params.crop_size == "auto":
    crop_size = adata.uns['meta']['spot_diameter_fullres']
else:
    crop_size = int(snakemake.params.crop_size)
    
for spot in tqdm.tqdm(adata.obs.itertuples()): #.head(5)
    cropd = crop_tile(slide, spot.x_pixel, spot.y_pixel, crop_size)
    
    pil_image = Image.fromarray(cropd, 'RGB')
    
    pil_image.save(os.path.join(snakemake.output.cropped_dir, '{bc}.jpg'.replace(
        "{bc}", spot.barcode)))

0it [00:00, ?it/s]